# 10a — Weight Initialization

Corresponds to: Note 10 — Weight Initialization

**Libraries introduced in this notebook:** No new libraries — numpy only, introduced in Note 03.

**What this notebook demonstrates:**
1. Xavier and He initialization formulas — computing σ for each layer of a concrete architecture
2. Activation variance trace — how variance evolves across layers under four initialization conditions
3. Symmetry demonstration — zero initialization keeps neurons identical; random initialization breaks symmetry

In [ ]:
import numpy as np

## Part 1: Xavier and He Initialization Formulas

The two initialization schemes differ only in whether fan-out is included and whether the numerator compensates for ReLU:

    Xavier:  Var(w) = 2 / (n_in + n_out)    →  σ = √(2 / (n_in + n_out))
    He:      Var(w) = 2 / n_in              →  σ = √(2 / n_in)

We compute σ for each layer of the architecture from Note 10 Part 4 and 5:
n₀=64, n₁=32, n₂=16, n₃=1 — ReLU hidden layers, linear output.
This matches the worked example in Exercise 10 Q10.

In [ ]:
layer_sizes = [64, 32, 16, 1]

print("Layer sizes:", layer_sizes)
print()
print(f"{'Layer':<8} {'n_in':<8} {'n_out':<8} {'He σ':<12} {'Xavier σ':<12} {'Used'}")
print("-" * 60)

for l in range(1, len(layer_sizes)):
    n_in  = layer_sizes[l - 1]
    n_out = layer_sizes[l]
    he_std    = np.sqrt(2 / n_in)
    xavier_std = np.sqrt(2 / (n_in + n_out))
    used = "He" if l < len(layer_sizes) - 1 else "Xavier"
    print(f"{l:<8} {n_in:<8} {n_out:<8} {he_std:<12.4f} {xavier_std:<12.4f} {used}")

Now initialize the weight matrices using the correct scheme for each layer and print their shapes.

In [ ]:
rng = np.random.default_rng(42)

weights = []
for l in range(1, len(layer_sizes)):
    n_in  = layer_sizes[l - 1]
    n_out = layer_sizes[l]
    if l < len(layer_sizes) - 1:
        std = np.sqrt(2 / n_in)                  # He — ReLU hidden layers
    else:
        std = np.sqrt(2 / (n_in + n_out))        # Xavier — linear output layer
    W = rng.normal(0, std, (n_in, n_out))
    weights.append(W)
    print(f"W[{l}]: shape={W.shape},  std used={std:.4f},  actual std={W.std():.4f}")

## Part 2: Activation Variance Trace

The core claim of Note 10 is that good initialization keeps activation variance stable across layers.
We test this by running a forward pass through an 8-layer ReLU network under four conditions:

    too_small:  σ = 0.01  — activations should collapse
    too_large:  σ = 1.0   — activations should explode
    xavier:     σ = √(2 / (n_in + n_out))
    he:         σ = √(2 / n_in)

Architecture: 8 hidden layers of 256 units each, input size 256.
We pass 1000 random inputs and record the std of activations at each layer.
This reproduces the logic behind the image in Note 10 Part 6.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def activation_stds(init_std_fn, n_layers=8, n_units=256, n_samples=1000, seed=0):
    """Run forward pass, return list of activation stds per layer."""
    rng = np.random.default_rng(seed)
    x = rng.normal(0, 1, (n_samples, n_units))
    a = x
    stds = []
    for l in range(n_layers):
        std = init_std_fn(n_units, n_units)
        W = rng.normal(0, std, (n_units, n_units))
        z = a @ W
        a = relu(z)
        stds.append(a.std())
    return stds

conditions = {
    "too_small" : lambda n_in, n_out: 0.01,
    "too_large" : lambda n_in, n_out: 1.0,
    "xavier"    : lambda n_in, n_out: np.sqrt(2 / (n_in + n_out)),
    "he"        : lambda n_in, n_out: np.sqrt(2 / n_in),
}

print(f"{'Layer':<8}" + "".join(f"{name:<14}" for name in conditions))
print("-" * 64)

results = {name: activation_stds(fn) for name, fn in conditions.items()}

for l in range(8):
    row = f"{l+1:<8}"
    for name in conditions:
        row += f"{results[name][l]:<14.4f}"
    print(row)

## Part 3: Symmetry Demonstration

Note 10 Part 2 proves algebraically that zero initialization causes permanent symmetry.
We demonstrate this concretely with a minimal network: 2 inputs, 2 hidden neurons, 1 output.

We run one forward pass and one backward pass under two conditions:
- Zero initialization — both hidden neurons start identically
- Random initialization — hidden neurons start differently

We then compare the gradients for the two hidden neurons to confirm the algebraic result.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward_and_grad(W1, W2, x, y_true):
    """One forward + backward pass. Returns gradient of loss w.r.t. W1."""
    # forward
    z1 = W1.T @ x                    # (2,)
    h  = sigmoid(z1)                 # (2,)
    z2 = W2.T @ h                    # scalar
    y_hat = sigmoid(z2)              # scalar

    # backward — binary cross-entropy loss
    dL_dy_hat = -(y_true / y_hat - (1 - y_true) / (1 - y_hat))
    dy_hat_dz2 = y_hat * (1 - y_hat)
    delta2 = dL_dy_hat * dy_hat_dz2  # scalar

    dL_dh = W2 * delta2              # (2,)
    dh_dz1 = h * (1 - h)            # (2,)
    delta1 = dL_dh * dh_dz1         # (2,)

    dL_dW1 = np.outer(x, delta1)    # (2, 2)
    return dL_dW1

x      = np.array([0.5, 0.8])
y_true = 1.0

# Zero initialization
W1_zero = np.zeros((2, 2))
W2_zero = np.zeros(2)
grad_zero = forward_and_grad(W1_zero, W2_zero, x, y_true)

print("Zero initialization:")
print(f"  Gradient for neuron 1: {grad_zero[:, 0]}")
print(f"  Gradient for neuron 2: {grad_zero[:, 1]}")
print(f"  Gradients identical: {np.allclose(grad_zero[:, 0], grad_zero[:, 1])}")
print()

# Random initialization
rng = np.random.default_rng(42)
W1_rand = rng.normal(0, np.sqrt(2/2), (2, 2))   # He init, n_in=2
W2_rand = rng.normal(0, np.sqrt(2/2), 2)
grad_rand = forward_and_grad(W1_rand, W2_rand, x, y_true)

print("Random (He) initialization:")
print(f"  Gradient for neuron 1: {grad_rand[:, 0]}")
print(f"  Gradient for neuron 2: {grad_rand[:, 1]}")
print(f"  Gradients identical: {np.allclose(grad_rand[:, 0], grad_rand[:, 1])}")

## Summary

This notebook demonstrated:

1. **Xavier and He formulas** — σ computed layer by layer for a [64, 32, 16, 1] architecture. He used for ReLU hidden layers; Xavier used for the linear output layer. Actual std of initialized matrices matches the target σ closely.
2. **Activation variance trace** — across 8 layers, too-small init (σ=0.01) collapses activations to near zero; too-large init (σ=1.0) causes explosion. Xavier and He keep std stable throughout. Reproduces the result shown in Note 10 Part 6.
3. **Symmetry demonstration** — with zero initialization, both hidden neurons produce identical gradients after one forward+backward pass, confirming the algebraic proof in Note 10 Part 2. With He initialization, gradients differ immediately.